## Load Libraries

In [3]:
import numpy as np
import pandas as pd 
import os
from typing import Optional, Dict, Any, Tuple, List
# import matplotlib.pyplot as plt  # 本地环境无matplotlib
# import seaborn as sns  # 本地环境无seaborn
from scipy import stats
import optuna
import warnings
import polars as pl
import time
import math
import gc
import lightgbm as lgb
from scipy.stats import spearmanr
from scipy.stats import pearsonr
import joblib
import kaggle_evaluation.default_inference_server
from sklearn.model_selection import TimeSeriesSplit
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="lightgbm")
from IPython.display import display, Markdown
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor, Pool
pd.set_option("display.max_rows", None)  
pd.set_option("display.max_columns", None)    

## Functions

In [4]:
# 导入工具模块
import sys
from pathlib import Path

# 添加toollab目录到Python路径
sys.path.insert(0, str(Path.cwd()))

from toollab import (
    FactorICAnalyzer, 
    FeaturePreprocessor, 
    FeatureEngineer,
    ModelTuner,
    describe_dataset,
    missing_duplicates_analysis,
    detect_outliers,
    timer,
    safe_spearman,
    time_series_cv_splits,
    sliding_window_cv_splits,
    true_online_cv_splits,
    analyze_factor_importance_from_model,
    analyze_lgbm_tree_complexity,  # 新增：树复杂度诊断
    create_tabular_mlp              # 新增：MLP模型创建
)

In [5]:
# ============================================================
# 【数据预处理】创建滞后特征
# ============================================================

def create_lagged_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    为数据集添加滞后的目标变量
    
    Args:
        dataframe: 原始数据框，包含 forward_returns, risk_free_rate, market_forward_excess_returns
    
    Returns:
        添加了3个滞后特征的数据框
    """
    df = dataframe.copy()
    
    # 要创建滞后特征的目标列
    target_columns = ['forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
    
    # 添加 is_scored 标记（训练集默认 False）
    if 'is_scored' not in df.columns:
        df['is_scored'] = False
    
    # 为每个目标列创建滞后特征
    for target_col in target_columns:
        if target_col in df.columns:
            df[f'lagged_{target_col}'] = df[target_col].shift(1)
            print(f"✅ 已创建 lagged_{target_col}")
        else:
            print(f"⚠️ 警告: 列 {target_col} 不存在，跳过")
    
    return df

print("✅ create_lagged_features 函数已定义")
print("   功能: 为 forward_returns, risk_free_rate, market_forward_excess_returns 创建滞后特征")
print("   用途: 在线学习时使用 lagged_forward_returns 作为上一期的真实target")

✅ create_lagged_features 函数已定义
   功能: 为 forward_returns, risk_free_rate, market_forward_excess_returns 创建滞后特征
   用途: 在线学习时使用 lagged_forward_returns 作为上一期的真实target


## FACTOR PREPROCESS

### CONFIG

In [12]:
# ============================================================
# 【核心配置】特征工程和模型训练配置
# ============================================================
 
TRAIN_PATH = 'train.csv'
LOCAL_GATEWAY_PATH = './'

# ============================================================
# 【在线学习配置】Online Learning Settings
# ============================================================
# 【1】滑动窗口大小（天数）
TRAIN_WINDOW_SIZE = 2000  # 使用最近2000天数据训练

# 【2】在线重训练频率
ONLINE_RETRAIN_FREQ = 1  # 每1次预测重训练一次模型

# 【3】时间衰减因子（用于样本加权）
TIME_DECAY_FACTOR = 0.995  # 最近数据权重=1.0，越老数据权重越低（2000天窗口适用）

# 【4】最小训练样本数
MIN_TRAIN_SAMPLES = 100  # 至少需要100个样本才能训练（2000天窗口适用）

# 【5】因子池大小
FACTOR_POOL_SIZE = 50  # 从98个原始因子中选50个稳定因子

# 【6】使用精简特征工程
USE_SLIM_FEATURES = True  # True=精简版(~2000特征)，False=完整版(~3500特征)

# 【7】是否使用特征工程（启用！）
USE_FEATURE_ENGINEERING = True  # True=50因子→扩展特征，False=直接使用50因子

# 【8】Validation配置（已弃用 - 使用Walk-Forward代替）
VALIDATION_SPLIT = 0.0035  # 已弃用
USE_CV = False              # 不使用交叉验证

# 【9】CV策略配置（已弃用，保留向后兼容）
USE_SLIDING_WINDOW_CV = False  # 在线学习模式不使用CV
USE_REVERSE_CV = False         
CV_STEP = 10                   
USE_CV_PRUNING = False         
CV_PRUNING_STEPS = [50, 200]   
CV_PRUNING_THRESHOLD = 0.04    
CV_TIME_DECAY = 0.995          

# 【10】因子分类配置
D_FACTOR_PREFIX = 'd'          # D类因子前缀（列名以'd'开头）
KEEP_ALL_D_FACTORS = True      # 保留全部D类因子

# 【11】因子预处理配置
USE_FACTOR_PREPROCESSING = True              # 启用因子预处理（IC翻转 + 三轮变换）
SKIP_TRANSFORM_FOR_D_FACTORS = True          # D类因子跳过Log/Rank变换（只做3-Sigma）

# 【12】Lagged特征配置
LAGGED_FEATURES = [
    'lagged_forward_returns',
    'lagged_risk_free_rate',
    'lagged_market_forward_excess_returns'
]

# 【13】Walk-Forward验证配置（模拟真实推理场景）
WALKFORWARD_SKIP_SAMPLES = 4000      # 跳过前4000个样本
WALKFORWARD_TRAIN_WINDOW = 2000      # 每个窗口用2000天训练
WALKFORWARD_VAL_WINDOW = 7           # 每个窗口验证7天
WALKFORWARD_STEP_SIZE = 7            # 窗口滑动步长（7天）
WALKFORWARD_TIME_DECAY = 0.995       # 窗口得分时间衰减因子

# ============================================================
# 【传统配置】Legacy Settings
# ============================================================
# 【注意】TOP_FEATURES_FOR_FE 已被 TOP_50_FACTORS 替代
# 在线学习模式下，使用RobustFactorSelector选出的50个稳定因子
TOP_FEATURES_FOR_FE = []  # 占位符，将被 TOP_50_FACTORS 替换

# 【2】滞后期: 创建历史值特征（精简模式）
LAG_PERIODS = [1, 3, 5, 7, 14, 20]  # 6个滞后期

# 【3】滚动窗口: 计算移动统计量（精简模式）
ROLLING_WINDOWS = [3, 6, 10, 20, 60]  # 5个窗口

# 【4】预测目标: 超额收益
TARGET = 'market_forward_excess_returns'

# 【5】要排除的列: 避免数据泄漏
COLS_TO_DROP = ['forward_returns', 'risk_free_rate', 'excess_return']

# 【6】Optuna超参数搜索配置（针对Walk-Forward验证）
TUNER_SEED = 2
N_TRIALS_LIGHTGBM = 30  # LightGBM trials
N_TRIALS_CATBOOST = 20  # CatBoost trials

# 【7】时间序列交叉验证折数（已弃用）
N_SPLITS = 1  # 不使用CV，设为1
USE_CATPOOL = False 
CAT_TASK = "CPU"

- Using date_id and TimeSeriesSplit ensures that training and validation sets respect temporal order, preventing data leakage.

- Optuna allows efficient search of the hyperparameter space using Bayesian optimization. Using Spearman correlation as the optimization metric aligns model objectives with rank-based trading signals.


In [14]:
# 加载训练数据
print(f"\n加载训练数据: {TRAIN_PATH}")
train = pd.read_csv(TRAIN_PATH)
print(f"✅ 训练集: {train.shape}")

# 创建滞后特征
print("\n创建滞后特征...")
train = create_lagged_features(train)

# 测试集（初始为空，后续在线学习时填充）
test = pd.DataFrame(columns=train.columns)
print(f"✅ 测试集: {test.shape} (空，待在线推理)")


加载训练数据: train.csv
✅ 训练集: (9021, 98)

创建滞后特征...
✅ 已创建 lagged_forward_returns
✅ 已创建 lagged_risk_free_rate
✅ 已创建 lagged_market_forward_excess_returns
✅ 测试集: (0, 102) (空，待在线推理)


In [ ]:
# ============================================================
# 【因子预处理】IC翻转 + 三轮变换优化
# ============================================================
# ⚠️ 注意：此cell需要在加载训练数据（train, test）之后运行
# 请在数据加载完成后，手动运行此cell

# 检查数据是否已加载
if 'train' not in locals() or 'test' not in locals():
    print("❌ 错误：请先加载 train 和 test 数据")
    print("   提示：请在 Training Pipeline 中加载数据后再运行此cell")
elif USE_FACTOR_PREPROCESSING:
    print("\n" + "="*80)
    print("【启动因子预处理】IC翻转 + 三轮变换")
    print("="*80)
    
    # 初始化
    analyzer = FactorICAnalyzer(window_size=20)
    preprocessor = FeaturePreprocessor(analyzer, target_col=TARGET)
    
    # Step 1: IC分析
    factor_df = analyzer.analyze_dataset(train, verbose=True)
    
    # Step 2: IC负值翻转
    train, test = preprocessor.flip_negative_ic_features(train, test, factor_df)
    negative_ic_features = factor_df[factor_df['IC'] < 0]['feature'].tolist()
    
    # Step 3: 三轮变换
    # Log变换
    train, test, log_report = preprocessor.apply_transformation_if_better(
        train, test, lambda x: np.sign(x) * np.log1p(np.abs(x)), '对数变换'
    )
    
    # Rank变换
    train, test, rank_report = preprocessor.apply_transformation_if_better(
        train, test, lambda x: x.rank(pct=True), 'Rank标准化'
    )
    
    # 3-Sigma裁剪
    train, test, sigma_report = preprocessor.apply_transformation_if_better(
        train, test, 
        lambda x: x.clip(x.mean() - 3*x.std(), x.mean() + 3*x.std()),
        '3-Sigma裁剪'
    )
    
    # ============================================================
    # 收集变换规则（用于后续验证集应用）
    # ============================================================
    preprocessing_rules = {
        'flip_rules': negative_ic_features,
        'transform_rules': {}
    }
    
    # 从三轮变换报告中提取改进的特征
    all_features = [c for c in train.columns if c.startswith(('A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'))]
    
    for feat in all_features:
        # 检查是否应用了Log变换
        if feat in log_report.index and log_report.loc[feat, '是否改进']:
            preprocessing_rules['transform_rules'][feat] = 'log'
        # 检查是否应用了Rank变换
        elif feat in rank_report.index and rank_report.loc[feat, '是否改进']:
            preprocessing_rules['transform_rules'][feat] = 'rank'
        # 检查是否应用了3-Sigma裁剪
        elif feat in sigma_report.index and sigma_report.loc[feat, '是否改进']:
            preprocessing_rules['transform_rules'][feat] = 'winsorize'
        else:
            preprocessing_rules['transform_rules'][feat] = 'none'
    
    print("\n✅ 因子预处理完成")
    print(f"   翻转因子: {len(preprocessing_rules['flip_rules'])}个")
    print(f"   变换规则: {len(preprocessing_rules['transform_rules'])}个")
else:
    print("⚠️ 因子预处理已禁用")

In [18]:
# ============================================================
# 【保存预处理规则】Save Preprocessing Rules
# ============================================================
# 目的：保存因子变换方式（非数据），用于后续验证集应用

import os
import json

REUSE_DIR = '/Users/curtis/Desktop/hull_tactical_prediction/v7_reuse'
os.makedirs(REUSE_DIR, exist_ok=True)

if USE_FACTOR_PREPROCESSING and 'preprocessing_rules' in locals():
    print("\n" + "="*80)
    print("【保存预处理规则】")
    print("="*80)
    
    # 保存JSON规则
    rules_path = os.path.join(REUSE_DIR, 'preprocessing_rules.json')
    with open(rules_path, 'w') as f:
        json.dump(preprocessing_rules, f, indent=2)
    
    print(f"✅ 预处理规则已保存: {rules_path}")
    print(f"   翻转规则: {len(preprocessing_rules['flip_rules'])}个")
    print(f"   变换规则: {len(preprocessing_rules['transform_rules'])}个")
    
    # 统计变换类型
    transform_types = {}
    for transform in preprocessing_rules['transform_rules'].values():
        transform_types[transform] = transform_types.get(transform, 0) + 1
    
    print("\n变换类型统计:")
    for t, count in sorted(transform_types.items(), key=lambda x: -x[1]):
        print(f"  {t}: {count}个")
    
    # 保存IC分析结果（如果存在）
    if 'factor_df' in locals():
        ic_path = os.path.join(REUSE_DIR, 'factor_ic_analysis.csv')
        factor_df.to_csv(ic_path, index=False)
        print(f"\n✅ IC分析结果已保存: {ic_path}")
    
    print(f"\n💡 后续可使用 apply_preprocessing_rules() 对验证集应用相同变换")
    
else:
    print("⚠️ 跳过保存（因子预处理未运行或preprocessing_rules未生成）")

⚠️ 跳过保存（因子预处理未运行或preprocessing_rules未生成）


In [24]:
# ============================================================
# 【实验3】动态因子池策略对比（真·动态重选）
# ============================================================
# 目的：对比固定因子池 vs 动态因子池（每个窗口都重选）
# ⚠️ 注意：需在因子预处理后、特征工程前运行

if 'train' not in locals():
    print("❌ 错误：train数据未加载")
else:
    print("\n" + "="*80)
    print("【实验3】固定因子池 vs 真·动态因子池对比")
    print("="*80)
    
    # 重新加载ModelTuner（获取新方法）
    import importlib
    import toollab.model_tuner
    importlib.reload(toollab.model_tuner)
    from toollab.model_tuner import ModelTuner
    
    # 准备完整训练数据（包含所有原始因子）
    exclude_meta = ['date_id', TARGET, 'forward_returns', 'risk_free_rate', 
                    'lagged_forward_returns', 'lagged_risk_free_rate', 
                    'lagged_market_forward_excess_returns', 'is_scored']
    
    # 获取所有因子列
    all_factor_cols = [c for c in train.columns if c not in exclude_meta]
    train_factors = train[all_factor_cols + [TARGET]].copy()
    
    print(f"\n原始因子数: {len(all_factor_cols)}")
    print(f"训练样本数: {len(train_factors)}")
    
    # 初始化tuner
    tuner = ModelTuner(
        seed=TUNER_SEED, 
        use_sliding_window=True, 
        train_window_size=90,  # 每个窗口90天
        cv_step=10             # 每10天滑动一次
    )
    
    # 简单模型参数（快速实验）
    simple_params = {
        'num_leaves': 31,
        'max_depth': 5,
        'learning_rate': 0.05,
        'n_estimators': 100,
    }
    
    # 对比实验：每个窗口都用最近60天IC重选因子
    print("\n开始对比实验...")
    print("  策略1: 固定因子池 - 全历史IC选因子，整个CV期间不变")
    print("  策略2: 真·动态因子池 - 每个窗口都用最近60天IC重选因子")
    
    compare_df = tuner.compare_dynamic_factor_selection_lgbm(
        train_data=train_factors,
        y=train_factors[TARGET],
        best_params=simple_params,
        top_n=30,           # 每次选30个因子
        ic_window_size=60   # 用最近60天计算IC
    )
    
    print("\n" + "="*80)
    print("【实验结果】固定 vs 真·动态因子池")
    print("="*80)
    print(compare_df.to_string(index=False))
    
    # 找出最佳策略
    best_strategy = compare_df.loc[compare_df['mean_spearman'].idxmax(), 'strategy']
    best_score = compare_df['mean_spearman'].max()
    print(f"\n✅ 最佳策略: {best_strategy} (mean_spearman={best_score:.4f})")
    
    print("\n💡 解读:")
    print("   - mean_spearman: 平均预测能力")
    print("   - std_spearman: 稳定性（越小越好）")
    print("   - stability_score: 综合得分（mean - 2*std，考虑风险调整）")
    print("\n   - 如果动态>固定: 说明因子需要每个窗口更新以适应市场变化")
    print("   - 如果固定>动态: 说明全局最优因子泛化能力更强")
    print("   - 动态策略会打印平均每个窗口更换多少个因子")


【实验3】固定因子池 vs 真·动态因子池对比

原始因子数: 94
训练样本数: 9021

开始对比实验...
  策略1: 固定因子池 - 全历史IC选因子，整个CV期间不变
  策略2: 真·动态因子池 - 每个窗口都用最近60天IC重选因子

[策略1] 固定因子池 - 使用全历史IC选因子...
  窗口600: 选中因子 -> ['P10', 'P8', 'M12']... (共30个)
  窗口700: 选中因子 -> ['M9', 'P8', 'E2']... (共30个)
  窗口800: 选中因子 -> ['D9', 'D3', 'D7']... (共30个)

  因子变化统计: 平均每个窗口更换 9.5 个因子（共30个）

【实验结果】固定 vs 真·动态因子池
                    strategy  n_windows  mean_spearman  std_spearman  stability_score
           fixed_all_history        893       0.133434      0.303913        -0.474391
dynamic_every_window(ic_60d)        893       0.119933      0.305581        -0.491230

✅ 最佳策略: fixed_all_history (mean_spearman=0.1334)

💡 解读:
   - mean_spearman: 平均预测能力
   - std_spearman: 稳定性（越小越好）
   - stability_score: 综合得分（mean - 2*std，考虑风险调整）

   - 如果动态>固定: 说明因子需要每个窗口更新以适应市场变化
   - 如果固定>动态: 说明全局最优因子泛化能力更强
   - 动态策略会打印平均每个窗口更换多少个因子


In [35]:
# ============================================================
# 【实验1】动态因子池 + 特征工程 + 树结构参数优化
# ============================================================
# 目的：在单个代表性窗口上深度分析最优树参数
# 流程：动态选因子 → 特征工程分裂 → 树参数网格搜索 → 树复杂度统计

if 'train' not in locals():
    print("❌ 错误：train数据未加载")
else:
    print("\n" + "="*80)
    print("【实验1】动态因子 + 特征工程 + 树参数网格搜索")
    print("="*80)
    
    # 重新加载ModelTuner（获取新方法）
    import importlib
    import toollab.model_tuner
    importlib.reload(toollab.model_tuner)
    from toollab.model_tuner import ModelTuner
    
    # 准备完整训练数据（包含所有原始因子）
    exclude_meta = ['date_id', TARGET, 'forward_returns', 'risk_free_rate', 
                    'lagged_forward_returns', 'lagged_risk_free_rate', 
                    'lagged_market_forward_excess_returns', 'is_scored']
    
    all_factor_cols = [c for c in train.columns if c not in exclude_meta]
    train_factors = train[all_factor_cols + ['date_id', TARGET]].copy()
    
    print(f"\n数据准备:")
    print(f"  原始因子数: {len(all_factor_cols)}")
    print(f"  训练样本数: {len(train_factors)}")
    
    # 选择代表性窗口（最后一个完整窗口）
    window_start_idx = len(train_factors) - 1220
    
    print(f"\n选择窗口: 样本 {window_start_idx}-{window_start_idx+90} (训练) + {window_start_idx+90}-{window_start_idx+100} (验证)")
    
    # 基础参数（非树结构参数）
    base_params = {
        'n_estimators': 200,
        'learning_rate': 0.05,
        'reg_lambda': 0.01,
        'reg_alpha': 0.01,
        'colsample_bytree': 0.8,
        'subsample': 0.8,
    }
    
    # 初始化tuner
    tuner = ModelTuner(seed=TUNER_SEED)
    
    # 执行完整pipeline
    print("\n开始实验...")
    exp1_results = tuner.experiment_dynamic_factors_with_feature_engineering(
        train_data=train_factors,
        window_start_idx=window_start_idx,
        window_size=1000,
        val_size=10,
        ic_window_size=60,
        top_n=30,
        base_params=base_params,
        max_depth_list=[3, 4, 5, 6],
        num_leaves_list=[8, 16, 31, 64],
        min_child_samples_list=[1, 3, 5, 10],
        min_split_gain_list=[0.0, 0.001, 0.01, 0.1]
    )
    
    print("\n" + "="*80)
    print("【实验结果】")
    print("="*80)
    
    print(f"\n动态选中因子: {exp1_results['selected_factors'][:5]}... (共{len(exp1_results['selected_factors'])}个)")
    print(f"特征工程后特征数: {exp1_results['n_features_after_engineering']}")
    
    print("\n最优树参数:")
    best_params = exp1_results['best_tree_params']
    print(f"  max_depth: {best_params['max_depth']}")
    print(f"  num_leaves: {best_params['num_leaves']}")
    print(f"  min_child_samples: {best_params['min_child_samples']}")
    print(f"  min_split_gain: {best_params['min_split_gain']}")
    print(f"  验证集Spearman: {best_params['spearman']:.4f}")
    
    print("\n树结构统计:")
    print(f"  总树数: {best_params['n_trees']}")
    print(f"  平均深度: {best_params['avg_depth']:.2f} / {best_params['max_depth_observed']} (设置:{best_params['max_depth']})")
    print(f"  平均叶子数: {best_params['avg_leaves']:.1f} / {best_params['max_leaves_observed']} (设置:{best_params['num_leaves']})")
    
    # 计算复杂度利用率
    depth_ratio = best_params['avg_depth'] / best_params['max_depth'] if best_params['max_depth'] > 0 else 0
    leaf_ratio = best_params['avg_leaves'] / best_params['num_leaves'] if best_params['num_leaves'] > 0 else 0
    print(f"\n复杂度利用率:")
    print(f"  深度利用率: {depth_ratio:.1%} (avg_depth / max_depth)")
    print(f"  叶子利用率: {leaf_ratio:.1%} (avg_leaves / num_leaves)")
    
    if depth_ratio < 0.5 or leaf_ratio < 0.5:
        print(f"\n⚠️ 警告: 树结构过于简单（利用率<50%），可能存在'树无法分裂'问题")
        print(f"   建议: 降低min_child_samples或min_split_gain")
    
    print("\nTop 10 参数组合:")
    print(exp1_results['grid_search_results'].head(10).to_string(index=False))
    
    print("\n💡 使用建议:")
    print(f"   - 如果需要更好的性能，尝试 max_depth={best_params['max_depth']}, num_leaves={best_params['num_leaves']}")
    print(f"   - 如果树利用率低，考虑放宽 min_child_samples 或 min_split_gain")
    print(f"   - 后续实验可使用这些最优参数作为base_params")


【实验1】动态因子 + 特征工程 + 树参数网格搜索

数据准备:
  原始因子数: 94
  训练样本数: 9021

选择窗口: 样本 7801-7891 (训练) + 7891-7901 (验证)

开始实验...
[2025-12-15 02:10:05] START experiment_dynamic_factors_with_feature_engineering

【实验1】动态因子选择 + 特征工程 + 树结构优化

窗口划分:
  数据总长度: 9021
  训练集: 样本 7801-8801 (1000天)
  验证集: 样本 8801-8811 (10天)

[Step 1/4] 动态因子选择 - 用最近60天IC选Top30因子...
  选中因子: ['S8', 'E3', 'P2', 'M9', 'D9']... (共30个)

[Step 2/4] 提取选中因子数据...
  训练集: (1000, 32)
  验证集: (10, 32)

[Step 3/4] 特征工程分裂 - 30因子 → 扩展特征...

【精简特征工程】输入: (1000, 32)
  创建Lag特征...
  创建Rolling特征...
  创建Pct_change特征...
  创建EWMA特征...
  创建交互特征...
  创建Rolling Z-score特征...
  合并特征...
  处理缺失值...
【精简特征工程】输出: (1000, 1252)
  生成特征数: 1250

【精简特征工程】输入: (10, 32)
  创建Lag特征...
  创建Rolling特征...
  创建Pct_change特征...
  创建EWMA特征...
  创建交互特征...
  创建Rolling Z-score特征...
  合并特征...
  处理缺失值...
【精简特征工程】输出: (10, 1252)
  生成特征数: 1250
  训练集扩展后: (1000, 1252)
  验证集扩展后: (10, 1252)

  特征工程后特征数: 1250
  X_train shape: (1000, 1250), y_train shape: (1000,)
  X_val shape: (10, 1250), y_val shape:

In [36]:
# ============================================================
# 【实验2】训练窗口大小网格搜索
# ============================================================
# 使用实验1的最优树参数，测试不同窗口大小的表现

if "exp1_results" not in locals():
    print("❌ 错误：请先运行实验1")
else:
    print("\n" + "="*80)
    print("【实验2】训练窗口大小对比")
    print("="*80)
    
    # 提取实验1的最优树参数
    best_tree = exp1_results["best_tree_params"]
    BEST_PARAMS = {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": int(best_tree["max_depth"]),
        "num_leaves": int(best_tree["num_leaves"]),
        "min_child_samples": int(best_tree["min_child_samples"]),
        "min_split_gain": float(best_tree["min_split_gain"]),
        "reg_lambda": 0.01,
        "reg_alpha": 0.01,
    }
    
    print(f"\n使用实验1最优树参数: max_depth={BEST_PARAMS['max_depth']}, num_leaves={BEST_PARAMS['num_leaves']}")
    
    # 准备全量特征数据（重用实验1选中的30个因子）
    selected_factors = exp1_results["selected_factors"]
    print(f"使用实验1选中的{len(selected_factors)}个因子")
    
    # 对全量数据做特征工程
    print("\n准备全量特征数据...")
    required_cols = selected_factors + ['date_id', TARGET]
    train_selected = train_factors[[c for c in required_cols if c in train_factors.columns]].copy()
    
    from toollab import FeatureEngineer
    engineer = FeatureEngineer(target_col=TARGET, verbose=False)
    train_expanded = engineer.create_features_slim(train_selected)
    
    # 提取特征列
    feature_cols = [c for c in train_expanded.columns if c not in ['date_id', TARGET]]
    X_full = train_expanded[feature_cols]
    y_full = train_expanded[TARGET]
    
    print(f"特征数: {len(feature_cols)}, 样本数: {len(X_full)}")
    
    # 窗口大小网格搜索
    print("\n开始窗口大小实验...")
    window_results = tuner.experiment_lgbm_window_sizes(
        X=X_full,
        y=y_full,
        best_params=BEST_PARAMS,
        window_sizes=[30, 60, 90, 120, 180, 240],
        step=10
    )
    
    print("\n" + "="*80)
    print("【实验结果】窗口大小对比")
    print("="*80)
    print(window_results.to_string(index=False))
    
    best_window = window_results.loc[window_results['stability_score'].idxmax(), 'train_window_size']
    best_score = window_results['stability_score'].max()
    print(f"\n✅ 最优窗口大小: {best_window}天 (stability_score={best_score:.4f})")
    print("\n💡 解读:")
    print("   - stability_score = mean_spearman - 2*std_spearman (风险调整后得分)")
    print("   - 窗口太小: 过拟合，std大")
    print("   - 窗口太大: 欠拟合，mean低")


【实验2】训练窗口大小对比

使用实验1最优树参数: max_depth=3, num_leaves=8
使用实验1选中的30个因子

准备全量特征数据...
特征数: 1250, 样本数: 9021

开始窗口大小实验...
[2025-12-15 02:16:14] START experiment_lgbm_window_sizes
[2025-12-15 02:50:03] DONE  experiment_lgbm_window_sizes (elapsed 2029.1s)

【实验结果】窗口大小对比
 train_window_size  n_windows  mean_spearman  std_spearman  stability_score  avg_tree_depth  avg_tree_leaves
               240        878       0.149766      0.313727        -0.477688        3.997655         7.173919
               180        884       0.147457      0.319974        -0.492491        3.989048         7.056150
               120        890       0.142617      0.324622        -0.506628        3.967074         6.842906
                90        893       0.133270      0.320696        -0.508121        3.954099         6.659432
                60        896       0.130704      0.325226        -0.519747        3.906882         6.221523
                30        899       0.098761      0.323366        -0.547971        3.

In [ ]:
# ============================================================
# 【特征工程】因子扩展
# ============================================================
# ⚠️ 注意：需在因子预处理之后运行

if 'train' not in locals():
    print("❌ 错误：train数据未加载")
elif USE_FEATURE_ENGINEERING:
    print("\n" + "="*80)
    print("【启动特征工程】因子扩展")
    print("="*80)
    
    # 初始化
    engineer = FeatureEngineer(target_col=TARGET, verbose=True)
    
    # 获取因子列（排除元数据列）
    exclude = ['date_id', TARGET, 'forward_returns', 'risk_free_rate']
    factor_cols = [c for c in train.columns if c not in exclude]
    
    print(f"\n原始因子数: {len(factor_cols)}")
    
    # 执行特征工程
    if USE_SLIM_FEATURES:
        # 精简版：~2000特征（适合在线学习）
        train_fe = engineer.create_features_slim(train)
        test_fe = engineer.create_features_slim(test)
    else:
        # 完整版：~3500特征
        train_fe = engineer.create_features(
            train, factor_cols, 
            lag_periods=LAG_PERIODS,
            rolling_windows=ROLLING_WINDOWS
        )
        test_fe = engineer.create_features(
            test, factor_cols,
            lag_periods=LAG_PERIODS,
            rolling_windows=ROLLING_WINDOWS
        )
    
    print(f"\n✅ 特征工程完成")
    print(f"   训练集: {train.shape} → {train_fe.shape}")
    print(f"   测试集: {test.shape} → {test_fe.shape}")
    
    # 更新数据
    train = train_fe
    test = test_fe
else:
    print("⚠️ 特征工程已禁用（USE_FEATURE_ENGINEERING=False）")

## Training Pipeline

In [ ]:
# ============================================================
# 【三模型集成 - 动态因子版】LightGBM + CatBoost + MLP（PyTorch）
# ============================================================
# 流程：
#   1. 用Optuna调参（每个窗口动态选因子+特征工程分裂）
#   2. 训练最终模型
#   3. 验证集评分并计算集成权重

import os
import torch
import torch.nn as nn

# ============================================================
# 【准备数据】使用预处理后的原始因子（未分裂）
# ============================================================
print("\n" + "="*80)
print("【数据准备】动态因子调参版本")
print("="*80)

# 注意：这里需要用预处理后的原始因子数据（94因子），而不是分裂后的特征
# 因为动态调参会在每个窗口重新做特征工程

# 如果你有保存的预处理数据
if 'train' in locals() and 'TARGET' in locals():
    # 检查train是否已经分裂过
    if len([c for c in train.columns if 'lag_' in c or 'roll' in c]) > 0:
        print("⚠️ 警告: 当前train数据已经包含分裂特征")
        print("  动态因子调参需要使用预处理后的原始因子数据")
        print("  建议重新加载或从预处理阶段的数据开始")
    else:
        print(f"✅ 使用预处理后的原始因子数据: {train.shape}")
        
        # 准备完整数据（包含target）
        train_data_full = train.copy()
        
        print(f"  数据形状: {train_data_full.shape}")
        print(f"  目标列: {TARGET}")
        print(f"  因子数: {len([c for c in train_data_full.columns if c not in ['date_id', TARGET, 'forward_returns', 'risk_free_rate', 'lagged_forward_returns', 'lagged_risk_free_rate', 'lagged_market_forward_excess_returns', 'is_scored']])}")
else:
    print("❌ 错误: 请先运行数据加载和预处理cell")
    
# 创建模型保存目录
os.makedirs('models', exist_ok=True)

print("\n💡 说明:")
print("  - 动态因子调参会在每个窗口重新选因子+分裂")
print("  - 不需要提前做90/10划分（调参时用滑动窗口CV）")
print("  - 调参完成后再用最优参数训练最终模型")


In [ ]:
# ============================================================
# 【Step 1】LightGBM动态因子调参（Optuna + 每窗口动态选因子+分裂）
# ============================================================
print("\n" + "="*80)
print("【Step 1/6】LightGBM动态因子超参数搜索")
print("="*80)

tuner_lgbm = ModelTuner(
    seed=TUNER_SEED,
    use_sliding_window=True,      # 启用滑动窗口CV
    train_window_size=90,          # 每个窗口90天训练
    cv_step=10,                    # 每10天滑动一次
    time_decay=0.995,              # 时间衰减因子（新样本权重高）
    use_pruning=False,             # 关闭剪枝（提高稳定性）
)

print(f"\n⚠️  完整版动态因子调参说明:")
print(f"  - 每个窗口都会:")
print(f"    1. 用最近60天IC动态选Top30因子")
print(f"    2. 对这30个因子做特征工程分裂（30 → ~1250特征）")
print(f"    3. 训练LightGBM并验证")
print(f"  - 预计窗口数: ~{(len(train_data_full) - 90) // 10}")
print(f"  - 每个trial需处理所有窗口")
print(f"  - 建议先用n_trials=5测试，确认流程正常后再增加")

# 使用固定树参数（可选，从实验1获得）
# 如果你运行过实验1，可以用这个：
# fixed_tree_params = {
#     'max_depth': 3,
#     'num_leaves': 8,
#     'min_child_samples': 1,
#     'min_split_gain': 0.0
# }
fixed_tree_params = None  # 让Optuna搜索所有参数

print(f"\n开始Optuna搜索（{N_TRIALS_LIGHTGBM} trials）...")
print(f"  滑动窗口: {tuner_lgbm.train_window_size}天训练 + 验证")
print(f"  IC窗口: 60天")
print(f"  选择因子数: 30个")

lgbm_study = tuner_lgbm.tune_lgbm_dynamic_full(
    train_data=train_data_full,
    target_col=TARGET,
    n_trials=N_TRIALS_LIGHTGBM,  # 建议先用5测试
    top_n=30,                     # 每窗口选30个因子
    ic_window_size=60,            # 用最近60天IC
    n_jobs=1,                     # 串行（避免内存问题）
    fixed_params=fixed_tree_params
)

best_lgbm_params = lgbm_study.best_params
print(f"\n✅ LightGBM最优参数:")
for k, v in best_lgbm_params.items():
    print(f"   {k}: {v}")
print(f"\n   最优CV得分: {lgbm_study.best_value:.6f}")

In [ ]:
# ============================================================
# 【Step 2】CatBoost动态因子调参（Optuna + 每窗口动态选因子+分裂）
# ============================================================
print("\n" + "="*80)
print("【Step 2/6】CatBoost动态因子超参数搜索")
print("="*80)

tuner_cat = ModelTuner(
    seed=TUNER_SEED,
    use_sliding_window=True,
    train_window_size=90,
    cv_step=10,
    time_decay=0.995,
    use_pruning=False,
)

print(f"\n开始Optuna搜索（{N_TRIALS_CATBOOST} trials）...")
print(f"  每个窗口动态选因子+分裂")

cat_study = tuner_cat.tune_catboost_dynamic_full(
    train_data=train_data_full,
    target_col=TARGET,
    n_trials=N_TRIALS_CATBOOST,  # 建议先用5测试
    top_n=30,
    ic_window_size=60,
    use_pool=USE_CATPOOL,
    task_type=CAT_TASK,  # "CPU" 或 "GPU"
    fixed_params=None
)

best_cat_params = cat_study.best_params
print(f"\n✅ CatBoost最优参数:")
for k, v in best_cat_params.items():
    print(f"   {k}: {v}")
print(f"\n   最优CV得分: {cat_study.best_value:.6f}")

In [ ]:
# ============================================================
# 【Step 3】MLP动态因子调参（Optuna + 每窗口动态选因子+分裂）
# ============================================================
print("\n" + "="*80)
print("【Step 3/6】MLP动态因子超参数搜索")
print("="*80)

tuner_mlp = ModelTuner(
    seed=TUNER_SEED,
    use_sliding_window=True,
    train_window_size=90,
    cv_step=10,
    time_decay=0.995,
    use_pruning=False,
)

print(f"\n⚠️  MLP动态因子调参说明:")
print(f"  - 和树模型一样，每个窗口都会:")
print(f"    1. 用最近60天IC动态选Top30因子")
print(f"    2. 对这30个因子做特征工程分裂（30 → ~1250特征）")
print(f"    3. 训练MLP并验证")
print(f"  - MLP也可以每个窗口适应不同的因子集合")

print(f"\n开始Optuna搜索（{N_TRIALS_LIGHTGBM} trials）...")
print(f"  搜索空间:")
print(f"    - hidden_layer_sizes: (64,) / (64,32) / (128,64)")
print(f"    - learning_rate_init: [1e-4, 3e-4, 1e-3, 3e-3]")
print(f"    - alpha: [1e-5, 1e-4, 1e-3, 1e-2]")

mlp_study = tuner_mlp.tune_mlp_dynamic_full(
    train_data=train_data_full,
    target_col=TARGET,
    n_trials=N_TRIALS_LIGHTGBM,  # 使用和LGBM一样的trial数
    top_n=30,                     # 每窗口选30个因子
    ic_window_size=60,            # 用最近60天IC
    n_jobs=1,
    fixed_params=None
)

best_mlp_params = mlp_study.best_params
print(f"\n✅ MLP最优参数:")
for k, v in best_mlp_params.items():
    print(f"   {k}: {v}")
print(f"\n   最优CV得分: {mlp_study.best_value:.6f}")


In [ ]:
# ============================================================
# 【Step 4】训练最终CatBoost模型（全部训练集）
# ============================================================
print("\n" + "="*80)
print("【Step 4/6】训练最终CatBoost模型")
print("="*80)

cat_model = tuner_cat.train_final_catboost(
    X=X_train,
    y=y_train,
    best_params=best_cat_params,
    model_path='models/final_catboost.cbm'
)

print(f"✅ CatBoost模型已保存: models/final_catboost.cbm")

In [ ]:
# ============================================================
# 【Step 5】训练PyTorch MLP模型（低频更新补充模型）
# ============================================================
print("\n" + "="*80)
print("【Step 5/6】训练PyTorch MLP模型")
print("="*80)

# MLP超参数（可以手动调优或用Optuna）
mlp_params = {
    'hidden_dims': [128, 64, 32],
    'dropout': 0.2,
    'lr': 0.001,
    'batch_size': 32,
    'epochs': 200,
}

print(f"\nMLP配置:")
print(f"  隐藏层: {mlp_params['hidden_dims']}")
print(f"  Dropout: {mlp_params['dropout']}")
print(f"  学习率: {mlp_params['lr']}")
print(f"  训练轮数: {mlp_params['epochs']}")

mlp_model = tuner_lgbm.train_final_mlp_tabular(
    X=X_train,
    y=y_train,
    best_params=mlp_params,
    model_path='models/final_mlp.pth'
)

print(f"\n✅ MLP模型已保存: models/final_mlp.pth")

In [ ]:
# ============================================================
# 【Step 6】验证集评分 + 计算集成权重
# ============================================================
print("\n" + "="*80)
print("【Step 6/6】验证集评分 + 集成权重计算")
print("="*80)

# 6.1 各模型预测
print("\n生成验证集预测...")

lgbm_val_pred = lgbm_model.predict(X_val)
cat_val_pred = cat_model.predict(X_val)

# MLP预测（PyTorch）
mlp_model.eval()
with torch.no_grad():
    X_val_tensor = torch.FloatTensor(X_val.values)
    mlp_val_pred = mlp_model(X_val_tensor).numpy().ravel()

# 6.2 计算各模型Spearman分数
print("\n计算验证集Spearman相关系数...")

score_dict = {
    'lgbm': safe_spearman(y_val, lgbm_val_pred),
    'catboost': safe_spearman(y_val, cat_val_pred),
    'mlp': safe_spearman(y_val, mlp_val_pred),
}

print("\n单模型验证集Spearman:")
print(f"  LightGBM:  {score_dict['lgbm']:.6f}")
print(f"  CatBoost:  {score_dict['catboost']:.6f}")
print(f"  MLP:       {score_dict['mlp']:.6f}")

# 6.3 计算竞赛官方评分（调整后夏普比率）
print("\n计算竞赛官方评分（Adjusted Sharpe）...")

official_scores = {
    'lgbm': calculate_score_metric(lgbm_val_pred, y_val.values),
    'catboost': calculate_score_metric(cat_val_pred, y_val.values),
    'mlp': calculate_score_metric(mlp_val_pred, y_val.values),
}

print("\n单模型官方评分:")
print(f"  LightGBM:  {official_scores['lgbm']:.6f}")
print(f"  CatBoost:  {official_scores['catboost']:.6f}")
print(f"  MLP:       {official_scores['mlp']:.6f}")

# 6.4 计算集成权重（基于Spearman，Temperature Scaling）
print("\n计算集成权重（Temperature Scaling）...")

scores_list = [score_dict['lgbm'], score_dict['catboost'], score_dict['mlp']]
weights = tuner_lgbm.compute_ensemble_weights_from_scores(scores_list)

ensemble_weights = {
    'lgbm': weights[0],
    'catboost': weights[1],
    'mlp': weights[2],
}

print("\n集成权重（基于验证集Spearman）:")
print(f"  LightGBM:  {ensemble_weights['lgbm']:.4f} ({ensemble_weights['lgbm']*100:.1f}%)")
print(f"  CatBoost:  {ensemble_weights['catboost']:.4f} ({ensemble_weights['catboost']*100:.1f}%)")
print(f"  MLP:       {ensemble_weights['mlp']:.4f} ({ensemble_weights['mlp']*100:.1f}%)")

# 6.5 集成预测
print("\n生成集成预测...")

pred_list = [lgbm_val_pred, cat_val_pred, mlp_val_pred]
ensemble_val_pred = tuner_lgbm.blend_predictions(pred_list, weights)

ensemble_spearman = safe_spearman(y_val, ensemble_val_pred)
ensemble_official = calculate_score_metric(ensemble_val_pred, y_val.values)

print("\n集成模型验证集表现:")
print(f"  Spearman:      {ensemble_spearman:.6f}")
print(f"  Official Score: {ensemble_official:.6f}")

# 对比提升
best_single_spearman = max(score_dict.values())
best_single_official = max(official_scores.values())

print("\n性能提升:")
print(f"  Spearman提升:  {(ensemble_spearman - best_single_spearman):.6f} " +
      f"({(ensemble_spearman / best_single_spearman - 1) * 100:+.2f}%)")
print(f"  Official提升:  {(ensemble_official - best_single_official):.6f} " +
      f"({(ensemble_official / best_single_official - 1) * 100:+.2f}%)")

# ============================================================
# 【保存集成配置】用于后续在线推理
# ============================================================
print("\n" + "="*80)
print("【保存集成配置】")
print("="*80)

ensemble_config = {
    'weights': ensemble_weights,
    'lgbm_params': best_lgbm_params,
    'catboost_params': best_cat_params,
    'mlp_params': mlp_params,
    'feature_cols': feature_cols,
    'validation_scores': {
        'lgbm_spearman': score_dict['lgbm'],
        'catboost_spearman': score_dict['catboost'],
        'mlp_spearman': score_dict['mlp'],
        'ensemble_spearman': ensemble_spearman,
        'ensemble_official': ensemble_official,
    }
}

joblib.dump(ensemble_config, 'models/ensemble_config.pkl')
print(f"✅ 集成配置已保存: models/ensemble_config.pkl")

print("\n" + "="*80)
print("【三模型集成完成】")
print("="*80)
print("\n可用模型:")
print("  1. models/final_lgbm.pkl       - LightGBM模型")
print("  2. models/final_catboost.cbm   - CatBoost模型")
print("  3. models/final_mlp.pth        - PyTorch MLP模型")
print("  4. models/ensemble_config.pkl  - 集成配置（权重+超参数）")

print("\n下一步:")
print("  - 使用集成模型进行在线推理")
print("  - 监控各模型在实际推理中的表现")
print("  - 根据需要动态调整集成权重")

In [ ]:
# ============================================================
# 【Step 3】训练最终LightGBM模型（全部训练集）
# ============================================================
print("\n" + "="*80)
print("【Step 3/6】训练最终LightGBM模型")
print("="*80)

lgbm_model = tuner_lgbm.train_final_lgbm(
    X=X_train,
    y=y_train,
    best_params=best_lgbm_params,
    model_path='models/final_lgbm.pkl'
)

print(f"✅ LightGBM模型已保存: models/final_lgbm.pkl")

# 树复杂度诊断
tree_stats = analyze_lgbm_tree_complexity(lgbm_model)
print(f"\n树结构统计:")
print(f"  总树数: {tree_stats['n_trees']}")
print(f"  平均深度: {tree_stats['avg_depth']:.2f} / {tree_stats['max_depth']}")
print(f"  平均叶子数: {tree_stats['avg_leaves']:.1f} / {tree_stats['max_leaves']}")

if tree_stats['avg_depth'] < 2.0:
    print(f"\n⚠️ 警告: 树深度过浅（{tree_stats['avg_depth']:.2f}），可能欠拟合")